# Graph-RAG Defect Reasoning — Demo

Thin demo notebook that drives the `lpbf_defect_reasoning` package (see `src/lpbf_defect_reasoning/`) end to end against the bundled sample data in `data/sample/`.

The pipeline logic itself lives in the package so it can be unit tested and reused outside a notebook — this notebook just wires it together and shows example output. See `README.md` and `CONTRIBUTING.md` for the package/version-control layout.

In [ ]:
# If running in Google Colab, uncomment the following to install the package:
# !pip install -q git+https://github.com/awanr032/ai-scientific-reasoning-lpbf-defects.git

# If running locally from a clone of this repo:
# !pip install -e "..[notebook]"


In [ ]:
from lpbf_defect_reasoning.evaluation import BENCHMARK, evaluate_graph_rag
from lpbf_defect_reasoning.generation import HFCausalLMAnswerer
from lpbf_defect_reasoning.indexing import load_default_embedder
from lpbf_defect_reasoning.io import load_chunks
from lpbf_defect_reasoning.pipeline import GraphRagPipeline
from lpbf_defect_reasoning.reporting import pretty_print_report


## Load sample data

`data/sample/graph_rag_chunks.json` holds pre-extracted literature chunks (text + relation triples + defect/parameter/mechanism labels).

In [ ]:
chunks = load_chunks("../data/sample/graph_rag_chunks.json")
print("Total chunks:", len(chunks))
print(chunks[0].keys())


## Build the pipeline

`GraphRagPipeline` builds the knowledge graph and FAISS semantic index from the chunks, and wraps the (pinned, see `src/lpbf_defect_reasoning/config.py`) generation LLM. Building the pipeline downloads/loads both pretrained models, so this cell can take a while and needs a GPU for the 7B generation model.

In [ ]:
embedder = load_default_embedder()
answerer = HFCausalLMAnswerer()  # loads the pinned Mistral-7B-Instruct model

pipeline = GraphRagPipeline(chunks, embedder=embedder, answerer=answerer)
print("Graph nodes:", pipeline.graph.number_of_nodes())
print("Graph edges:", pipeline.graph.number_of_edges())
print("Semantic index size:", len(pipeline.index))


## Example query (retrieval + grounded answer)

In [ ]:
answer, evidence = pipeline.qa(
    "Why does high laser power lead to keyhole porosity in LPBF?"
)

print("=== Retrieved Evidence ===")
for r in evidence:
    print(r["chunk_id"], "| defects:", r.get("possible_defects"))

print("\n=== ANSWER ===")
print(answer)


## Benchmark evaluation

Re-run this after any change to the graph, index, or pinned models to check retrieval quality hasn't regressed (see `lpbf_defect_reasoning.evaluation`).

In [ ]:
df = evaluate_graph_rag(pipeline.qa, BENCHMARK, k=5)
df


In [ ]:
print(df[[
    "retrieval_accuracy",
    "defect_label_precision",
    "defect_label_recall",
    "latency",
]].mean())

print("mean parameter_accuracy:", df["parameter_accuracy"].dropna().mean())


## Agent-based reasoning (classification + graph paths + structured report)

In [ ]:
report = pipeline.agent(
    "Why does high laser power lead to keyhole porosity in LPBF?"
)
pretty_print_report(report)
